# Accessing GNSS Observations with the EarthScope SDK

**Version:** 1.0 | **Last updated:** 2026-07-09 

**Author:** Eshanta Mishra | **Author institution:** EarthScope Consortium

**Maintainer:** EarthScope OnRamp Team | **Maintainer's contact :** help@earthscope.org

**Estimated Time:**  ~ 30 minutes| **Pathway:** MVP1

**License:** CC-BY-4.0

## Introduction

**What this notebook does:** It instantiates the EarthScope SDK and allows you to retrieve GNSS observations for your station of interest.

**Why it is useful:** This notebook provides a hands-on on how to use the SDK to retrieve GNSS observations from the cloud and load it into a dataframe. It provides the users an insight into how modern workflows can be more efficient compared to traditional methods of downloading and accessing GNSS observations as RINEX files.

**What you will accomplish:** By the end of this notebook, we will have accomplished:
* instantiating the EarthScope SDK client
* Select data for stations using the station name and time range.
* Retrieve observations and load into dataframes.
* Filter (slice) the GNSS data based on different parameters.

---

### Prerequisites


Before starting this notebook, you should:
* [ ] This is an introductory notebook. Being familiar with **Geodesy, GNSS, python dataframes** is recommended but not required.

---

### GeoLab Compute Resources

| Setting | Recommended |
|---|---|
| **Image** | GeoLab (default image)|
| **Server size** | 4 GB RAM, ~0.5 CPUs (default server) |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the basics of GNSS data available through the EarthScope SDK.
2. Use the EarthScope SDK client to retreive GNSS data from EarthScope.

## Relevant Documentation & Resources

* [EarthScope SDK documentation](https://docs.earthscope.org/sdk)

## Contents

1. [Basics of GNSS](#id-1-basics-of-gnss)
2. [Setup & Imports](#id-2-setup-imports)
3. [Retrieve Observations for a Single Station](#id-3-retrieve-observations-for-a-single-station)
4. [Understanding the Observation fields](#id-4-understanding-the-observation-fields)
5. [Requesting Only the Data you Need](#id-5-requesting-only-the-data-you-need)
6. [Larger than Memory Requests with Query Plans](#id-6-larger-than-memory-requests-with-query-plans)
7. [Exploration Exercises](#id-7-exploration-exercises)
8. [Troubleshooting & Support](#id-8-troubleshooting-support)

## 1. Basics of GNSS

A Global Navigation Satellite System (GNSS) is a constellation of satellites that broadcast timing signals. GNSS includes constellations such as GPS (United States), GLONASS (Russia), BeiDou (China) etc. A ground station (receiver plus antenna) records the data transmitted by these satellites several times per minute.

Each of those recordings is a GNSS observation. It is a raw measurement of the signal at a given instant for a given satellite, on one frequency. Some commonly used observations are:

1. **Pseudorange**: Apparent distance to the satellite, in meters. It is called *pseudo* because the receiver and satellite clocks aren't perfectly synchronized.
2. **Carrier phase**: It is a more precise distance measurement between the satellite and the ground, counted in whole cycles of the carrier wave (with an unknown starting offset)
3. **SNR**: Signal-to-Noise Ratio. It is a parameter that tells how strong and clean the received signal is.

**Raw Observations vs. derived products**

The data we retrieve in this notebook are raw observations. The geodetic results you may ultimately want such as a station's position, displacement over time, etc., are derived products. These derived products are computed by processing many observations together. 

## 2. Setup & Imports

In [ ]:
import datetime as dt
import polars as pl
import os

from earthscope_sdk import AsyncEarthScopeClient

### Instantiating the EarthScope SDK

#### Authentication

Since you use your EarthScope account to log into GeoLab, your EarthScope credentials are already available inside it. The client finds your credentials automatically. This removes the step of logging in again using the CLI or passing tokens manually, as you would do in a non-GeoLab Environment. If you are running this notebook outside geolab, you will need to authenticate using the EarthScope CLI (detailed instructions for doing this can be found [here](https://gitlab.com/earthscope/public/earthscope-cli)).

#### Async client

`AsyncEarthScopeClient` is the asynchronous client. [Async](https://docs.earthscope.org/sdk/usage#async-usage) lets the SDK process a large query into many concurrent sub-requests, making large quantity of data retrieval more efficient. The SDK also comes with a synchronous `EarthScopeClient` if you prefer. The methods used below are identical on the synchronous client, just remove the `async` / `await` keywords. See [Sync-vs-Async programming](https://www.geeksforgeeks.org/javascript/synchronous-and-asynchronous-programming/) for more information on sync and async programming principles.

In [ ]:
es = AsyncEarthScopeClient()

### Configuration

Set your parameters here before running the rest of the notebook. Every subsequent cells read from these variables. So, this is the only place you need to edit to point the notebook at different data.

In [ ]:
# Modify these values before running the notebook.

STATION = "AC6000USA"                  # GNSS station (9-character ID)
SESSION = "A"                          # Session name
START = dt.datetime(2025, 7, 20, 21)   # Query start (UTC)
END = dt.datetime(2025, 7, 21, 3)      # Query end (UTC)

OUTPUT_DIR = os.path.join(os.environ["SCRATCH_BUCKET"], "mvp1-geodesy-nb1-output") # output folder on personal scratch space on S3

print(f"Configuration set. Outputs will be saved to: {OUTPUT_DIR}")

## 3. Retrieve Observations for a single station

In this step, we will retrieve GNSS observations for a single station and time window as an Apache Arrow table through the [EarthScope API](https://api.earthscope.org/beta/docs#get-/data-products/gnss/observations).

[Arrow](https://arrow.apache.org/) is a fast, columnar, in-memory format that most dataframe libraries read with little or no copying. The API also lets you request an arbitrary time window, unlike the daily data chunks of RINEX files (RINEX provides one file per station per UTC day).

The expected output is a single table with one row per satellite, per signal, per epoch

In [ ]:
#describe the request, then .fetch() to run the query.
table = await es.data.gnss_observations(
    start_datetime=START,
    end_datetime=END,
    station_name=STATION,
    session_name=SESSION,
).fetch()

table

Convert the Arrow table to a [Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html) dataframe with `pl.from_arrow(...)`. This is zero-copy, so it is very efficient. Sorting by `timestamp` makes the rows read chronologically.

In [ ]:
df = pl.from_arrow(table).sort("timestamp")
df

## 4. Understanding the Observation fields

The default data query returns every field - the same information you would find in a RINEX observation file. Let us take a moment to understand the data structure.
Each row is a single measurement (one satellite, one signal, one instant). Inspect the columns, the constellations present, and the range of signal strengths.

In [ ]:
# Inspect structure and coverage
print(df.schema)                                    
print("Constellations:", df["system"].unique().sort().to_list())
print("Observation codes:", df["obs_code"].unique().sort().to_list())

### What each field means

| Column | Type | Meaning |
|---|---|---|
| `timestamp` | datetime (UTC) | The epoch of the observation. |
| `satellite` | int | Satellite number within its constellation (its PRN / slot). Combine with `system` for a globally unique ID. |
| `obs_code` | str | Which signal was measured, e.g. `1C`, `2W`, `2L` (decoded below). |
| `range` | float | Pseudorange in meters. It is the apparent satellite to receiver distance. |
| `phase` | float | Carrier phase in cycles. It is used to drive a precise but ambiguous range measurement. |
| `snr` | float | Signal strength as carrier-to-noise density ($C/N_0$), roughly in dB-Hz. Higher values correspond to stronger signal. |
| `slip` | int | Cycle-slip indicator; `null` when phase tracking was continuous. |
| `flags` | int | Per-observation status flags (e.g. loss-of-lock). |
| `fcn` | int | Frequency channel number — used by GLONASS's FDMA signals; `0` otherwise. |
| `system` | str | Constellation code (see below). |
| `igs` | str | The station's IGS long name (e.g. `AC6000USA`). |

### Reading the `system` column

| Code | Constellation |
|---|---|
| `G` | GPS (USA) |
| `R` | GLONASS (Russia) |
| `E` | Galileo (EU) |
| `C` | BeiDou (China) |
| `J` | QZSS (Japan) |
| `I` | NavIC / IRNSS (India) |
| `S` | SBAS (augmentation) |

### Reading the `obs_code` column

An observation code names a specific signal as **band + tracking attribute**:

* The **digit** is the frequency band: `1` (L1), `2` (L2), `5` (L5), and so on.
* The **letter** is the tracking mode or signal component: `C` (C/A or civil code), `W` (semi-codeless Z-tracking of the encrypted P-code), and `L` / `Q` / `X` / `I` (specific modern civil components).

For example, `1C` is the classic GPS L1 C/A signal, `2W` is L2 P(Y) via Z-tracking, `2L` is the L2C signal.

## 5. Requesting Only the Data You Need

When you are working with RINEX files, the files include everything: all fields, constellations, satellites etc. Many analyses only need a small slice of these data. Passing filters to `gnss_observations()` in the SDK pushes that selection to the server, so only the data that you asked for is transferred.

The payoff is immediate: requesting a single signal from a single satellite over two months returns in a fraction of the time and requires less memory, where full RINEX for the same window would be orders of magnitude larger and mostly discarded. You also skip RINEX parsing entirely because the results arrive as an Arrow table, rather than a set of daily text files you have to decode first.

*A smaller slice also means less data moved across the network, which keeps costs down on EarthScope's side.*

The resulting output is a dataframe containing only the requested stations, constellations, satellites, signals and fields. For example, only `snr` and `range` columns when `field=["snr","range"]`.

Each filter below is optional, accepts a single value or a list, and shrinks the request:

* `station_name`: one name or a list of names
* `network_name`: request a whole network at once
* `system`: constellation code(s), e.g. `"G"` or `["G", "R"]`
* `satellite`: specific satellite number(s)
* `obs_code`: specific signal(s), e.g. `["1C", "2L"]`
* `field`: which measurement column(s) to return, e.g. `["snr", "range"]`


In [ ]:
start = dt.datetime(2025, 7, 20)
end = start + dt.timedelta(days=7)

table = await es.data.gnss_observations(
    start_datetime=start,
    end_datetime=end,
    station_name=["P717", "P453", "P146", "P147", "P041"],
    session_name="A",
    system=["G", "R"],
    obs_code=["1C", "2L"],
    satellite=["7", "21", "28"],
    field=["snr", "range"],
).fetch()

df_sliced = pl.from_arrow(table).sort("timestamp")
df_sliced

### Saving your results

Fetching takes time, so it's worth writing results to disk once you have data you'll reuse. In GeoLab, where you write matters:

| Location | Path | Persistence | Use it for |
|---|---|---|---|
| **Home** | `/home/jovyan/` | Private, persistent, 50 GB limit | Notebooks, scripts, and results you want to keep while working on your project |
| **Shared** | `/home/jovyan/shared/` | Read-only | Datasets and starter notebooks placed there by instructors. You can copy files out, but not modify the original |
| **Scratch** | `/earthscope-scratch/` (exact path in `os.environ["SCRATCH_BUCKET"]`) | Temporary and auto-deletes after ~2 weeks | Large intermediates over the 50 GB home limit |

Use Parquet rather than CSV. It's columnar, compressed, and preserves data types — your `timestamp` comes back as a datetime and `snr` as a float, with no re-parsing.

In [ ]:
# Step description: Save the sliced dataframe to your Scratch directory.
out_path = os.path.join(OUTPUT_DIR, "sliced_observations.parquet")
df_sliced.write_parquet(out_path)

print(f"Wrote {len(df_sliced):,} rows to {out_path}")

## 6. Larger than Memory Requests with Query Plans

### What is a query plan?

**What it does**: Instead of calling `.fetch()` which returns one big table, you build a query plan by leaving `.fetch()` off. A query plan is iterable, i.e. it gives the result set in manageable groups. This allows you to process one group at a time.

**Why it matters**: Some requests such as an entire network for a week will not fit in memory at once. Query plans also limit how many requests hit the API at the same time. Each group's sub-requests run in parallel and are collected into a single table you can process before moving on.

The expected output is a summary for each group (row count, time span, and stations), rather than one huge dataframe.

In [ ]:
# Build a query plan (note: no .fetch())
start = dt.datetime(2025, 7, 20)
end = start + dt.timedelta(days=7)

plan = es.data.gnss_observations(
    start_datetime=start,
    end_datetime=end,
    network_name="PERM:Alaska",
    session_name="A",
    system="G",
    field=["phase", "range", "snr"],
)
await plan.plan()
print(plan)  # preview the plan's request/group counts before any data is fetched

def summarize(table):
    # Print a quick summary of one group's table.
    d = pl.from_arrow(table)

    # Mean SNR per station for this group: which sites have better signals?
    snr_by_station = (
        d.group_by("igs")
        .agg(pl.col("snr").mean().round(1).alias("mean_snr"))
        .sort("mean_snr", descending=True)
    )

    # Other things you might compute:
    # - Weak signals that may indicate obstruction or hardware trouble
    #     d.filter(pl.col("snr") < 25).group_by("igs").len()
    # - Append each group to disk and process the whole week later
    #     # d.write_parquet(f"{OUTPUT_DIR}/{d['timestamp'].min():%Y%m%d}.parquet")
    # -------------------------------------------------------------------
    print(f"{d['timestamp'].min()} → {d['timestamp'].max()}")
    print(snr_by_station)

Processing the plan one group at a time. Each group is fetched, processed, and released before the next begins, so the full week never sits in memory at once.

In [ ]:
async for table in plan.group_by_day():
    summarize(table)

Processing the plan one station at a time across the whole window instead.

In [ ]:
async for table in plan.group_by_station():
    summarize(table)

 > **Check**: Each timestamp range and table represents one group.

### A rough guide for choosing a strategy

* Less than a week: `.fetch()` the whole results at once.
* Weeks to months, or many stations: iterate with `group_by_day()` (or `group_by_station()`)
* Months to years: define custom batches with `plan.group_by()`.

For custom grouping, request ordering and performance details (concurrency, rate limiting, and retries), see the [EarthScope SDK documentation](https://docs.earthscope.org/sdk/query-plans#option-4-custom-grouping-advanced). The SDK applies no size limits by default. For very large queries, you can also cap memory or time (more information available in the same documentation).

## 7. Exploration Exercises

Now that you've completed the core workflow, try modifying the parameters below to explore how the results change.

**Try these modifications:**

1. **Change the station:** Set `STATION` in the Configuration section to a different 9-character station ID and re-run Section 3. Does the station report the same constellations?

2. **Isolate one signal:** In Section 5, request a single `obs_code` (e.g. `"2L"`) with `field="snr"`. How much smaller is the result?

3. **Save a different result:** You saved `df_sliced` above. Do the same for the full dataframe `df` from Section 3, giving it a different filename. Compare the two file sizes — how much did filtering the request save you?

In [ ]:
# Exploration cell — use this space to experiment

## 8. Troubleshooting & Support

### Further Resources

* [EarthScope SDK Documentation](https://docs.earthscope.org/sdk)
* [Authentication using EarthScope CLI](https://gitlab.com/earthscope/public/earthscope-cli)(For non-GeoLab environments)
* [SDK GNSS Observation tutorial](https://docs.earthscope.org/sdk/gnss-obs-tutorial)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [GeoLab Community Forum](https://earthscope.discourse.group/latest)
* [Sync-vs-Async programming](https://www.geeksforgeeks.org/javascript/synchronous-and-asynchronous-programming/)